# ITCHI real-data inspection notebook

This notebook is a step-by-step inspection workflow for the real-data ITCHI pipeline.

It is intentionally diagnostic rather than fully automated. The goal is to inspect:

1. IBTrACS-derived `track_df`;
2. ROCLOUD-derived `rocloud_df`;
3. MSWEP precipitation snapshots;
4. longitude/latitude grids;
5. selected target times;
6. the first snapshot input;
7. compiled event metadata;
8. spatial ITCHI products.

**Scientific rule:** MSWEP precipitation is treated as a valid-time snapshot field. This notebook does not sum, accumulate or resample precipitation through time.


In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
import xarray as xr

from itchi.compiler import compile_itchi_event
from itchi.ibtracs import read_ibtracs_track_dataframe
from itchi.io import write_compiled_event
from itchi.mswep import (
    get_mswep_lon_lat,
    prepare_mswep_precipitation,
    read_mswep_dataset,
    subset_mswep_domain,
)
from itchi.precipitation_snapshots import is_synoptic_time
from itchi.rocloud import rocloud_text_to_dataframe
from itchi.snapshot_inputs import build_event_snapshot_inputs_from_tables


## 1. User configuration

Edit these paths and parameters for your local or HPC environment.

For an initial end-to-end smoke run, `q90`, `q95` and `q99` may be scalars. For scientific production, they should be gridded climatological thresholds compatible with MSWEP.


In [ ]:
# ============================================================
# Input paths
# ============================================================

IBTRACS_PATH = Path("data/ibtracs/ibtracs.nc")
ROCLOUD_PATH = Path("data/rocloud/EP_TCSize_2000_2024.dat")
MSWEP_PATH = Path("data/mswep/event_precip.nc")

# ============================================================
# Event configuration
# ============================================================

STORM_ID = "EP182023"
START_TIME = "2023-10-24T00:00:00"
END_TIME = "2023-10-25T06:00:00"

# MSWEP variable name. Use None only if the file has one unambiguous variable.
PRECIPITATION_VARIABLE = "precipitation"

# Optional spatial subset. Set all to None to avoid subsetting.
LON_MIN = -110.0
LON_MAX = -90.0
LAT_MIN = 5.0
LAT_MAX = 30.0
PAD_DEGREES = 2.0

# Initial scalar thresholds for smoke testing.
# Replace with gridded DataArrays for scientific production.
q90 = 10.0
q95 = 20.0
q99 = 30.0

OUTPUT_PATH = Path(f"outputs/events/{STORM_ID}_itchi.nc")


## 2. Build target times

The event is evaluated at 6-hourly synoptic times. This is a selection schedule, not a precipitation accumulation rule.


In [ ]:
target_times = list(pd.date_range(start=START_TIME, end=END_TIME, freq="6h"))

if not target_times:
    raise ValueError("No target times were generated.")

non_synoptic = [time for time in target_times if not is_synoptic_time(time)]

if non_synoptic:
    raise ValueError(f"Generated non-synoptic target times: {non_synoptic}")

target_times


## 3. Read IBTrACS track information

The IBTrACS adapter converts one storm into the internal `track_df` contract used by `snapshot_inputs.py`.


In [ ]:
track_df = read_ibtracs_track_dataframe(
    path=IBTRACS_PATH,
    storm_id=STORM_ID,
)

track_df.head()


In [ ]:
track_df.loc[
    (track_df["time"] >= pd.Timestamp(START_TIME))
    & (track_df["time"] <= pd.Timestamp(END_TIME))
]


## 4. Read ROCLOUD database

The ROCLOUD adapter supports the block-based database format:

```text
storm_id, storm_name, declared_entries,
date hh lat lon MWS CPSL RNE RNO RSO RSE Rp A D S RBP_RNE RBP_RNO RBP_RSO RBP_RSE RBP_mean
```

The parser maps `RNO -> rocloud_rnw` and `RSO -> rocloud_rsw`.


In [ ]:
rocloud_df = rocloud_text_to_dataframe(
    path=ROCLOUD_PATH,
    synoptic_only=True,
    validate_record_counts=False,
)

rocloud_df.head()


In [ ]:
rocloud_df.loc[
    (rocloud_df["storm_id"] == STORM_ID)
    & (rocloud_df["time"] >= pd.Timestamp(START_TIME))
    & (rocloud_df["time"] <= pd.Timestamp(END_TIME))
]


## 5. Read and subset MSWEP

The MSWEP adapter opens the NetCDF file, optionally subsets the spatial domain, extracts the precipitation variable and filters existing snapshots to synoptic hours.

No temporal accumulation is performed.


In [ ]:
mswep_ds = read_mswep_dataset(MSWEP_PATH)

mswep_ds


In [ ]:
if all(value is not None for value in [LON_MIN, LON_MAX, LAT_MIN, LAT_MAX]):
    mswep_ds = subset_mswep_domain(
        mswep_ds,
        lon_min=LON_MIN,
        lon_max=LON_MAX,
        lat_min=LAT_MIN,
        lat_max=LAT_MAX,
        pad_degrees=PAD_DEGREES,
    )

mswep_ds


In [ ]:
precipitation = prepare_mswep_precipitation(
    data=mswep_ds,
    precipitation_variable=PRECIPITATION_VARIABLE,
    synoptic_only=True,
)

precipitation


In [ ]:
precipitation.sel(valid_time=slice(START_TIME, END_TIME))


## 6. Extract longitude and latitude grids

The pipeline expects longitude and latitude grids aligned with the precipitation field.


In [ ]:
lon, lat = get_mswep_lon_lat(precipitation)

lon, lat


## 7. Visual check of one precipitation snapshot

This plot is diagnostic. It should be used to confirm that the selected domain and time are physically reasonable before compiling ITCHI.


In [ ]:
snapshot_for_plot = precipitation.sel(valid_time=target_times[0])

plt.figure(figsize=(7, 5))
snapshot_for_plot.plot()
plt.title(f"MSWEP precipitation snapshot: {target_times[0]}")
plt.show()


## 8. Build ITCHI snapshot inputs

This step combines precipitation, thresholds, track metadata, ROCLOUD radii, lon/lat grids and configuration into dictionaries compatible with the snapshot pipeline.


In [ ]:
snapshot_inputs = build_event_snapshot_inputs_from_tables(
    precipitation_data=precipitation,
    q90=q90,
    q95=q95,
    q99=q99,
    lon=lon,
    lat=lat,
    track_df=track_df,
    rocloud_df=rocloud_df,
    storm_id=STORM_ID,
    target_times=target_times,
    precipitation_variable=None,
    precipitation_time_coord="valid_time",
    precipitation_method="exact",
    precipitation_tolerance=None,
    track_tolerance_hours=None,
    rocloud_tolerance_hours=None,
    r34_unit="nm",
    rocloud_unit="km",
    fallback_direct_radius_km=None,
    radius_fill_strategy="mean_available",
    wind_below_threshold_mode="relative_to_vmax",
    run_quality_control=True,
)

len(snapshot_inputs), snapshot_inputs[0].keys()


In [ ]:
first_input = snapshot_inputs[0]

{
    "center_lon": first_input["center_lon"],
    "center_lat": first_input["center_lat"],
    "r34_by_quadrant": first_input["r34_by_quadrant"],
    "rocloud_by_quadrant": first_input["rocloud_by_quadrant"],
    "vmax_kt": first_input["vmax_kt"],
    "rmw_km": first_input["rmw_km"],
}


## 9. Compile event

The compiler runs the snapshot pipeline for each target time, stacks spatial fields and computes event-level products such as `ITCHI_max` and `ITCHI_acc`.


In [ ]:
compiled = compile_itchi_event(
    snapshot_inputs=snapshot_inputs,
    time_values=target_times,
    run_snapshot_quality_control=True,
)

compiled.keys()


In [ ]:
compiled["metadata"]


## 10. Inspect ITCHI products

The following plots are first-pass diagnostics for the final gridded fields.


In [ ]:
itchi = compiled["snapshots"]["ITCHI"]
itchi_max = compiled["event_products"]["ITCHI_max"]
itchi_acc = compiled["event_products"]["ITCHI_acc"]

float(itchi.min()), float(itchi.max()), float(itchi_max.max()), float(itchi_acc.max())


In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4), constrained_layout=True)

itchi.isel(time=0).plot(ax=axes[0])
axes[0].set_title(f"ITCHI first snapshot\n{target_times[0]}")

itchi_max.plot(ax=axes[1])
axes[1].set_title("ITCHI_max")

itchi_acc.plot(ax=axes[2])
axes[2].set_title("ITCHI_acc")

plt.show()


## 11. Export compiled event

The compiled event is exported as NetCDF or Zarr depending on the extension of `OUTPUT_PATH`.


In [ ]:
OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)

written_path = write_compiled_event(
    compiled_event=compiled,
    path=OUTPUT_PATH,
    attrs={
        "title": "Real-data ITCHI notebook output",
        "storm_id": STORM_ID,
        "itchi_version": "0.1",
        "precipitation_source": "MSWEP NetCDF",
        "track_source": "IBTrACS NetCDF",
        "rocloud_source": "ROCLOUD text database",
        "precipitation_treatment": "snapshot_not_accumulation",
        "start_time": str(target_times[0]),
        "end_time": str(target_times[-1]),
    },
    overwrite=True,
)

written_path


## 12. Next scientific checks

After the notebook runs successfully, inspect:

- whether the storm center lies inside the MSWEP subset for every target time;
- whether `R_direct_q <= ROCLOUD_q` is always satisfied;
- whether missing R34 or ROCLOUD quadrants were filled;
- whether `ITCHI_max` spatial maxima occur in physically plausible regions;
- whether scalar thresholds should be replaced by gridded monthly/hourly climatology fields.
